# Notebook

Backup notebook to ensure saving of certain variables can be tested.

In [12]:
%load_ext autoreload
%autoreload 2
from main import run_experiment
from collections import Counter
import pickle
import pandas as pd

The autoreload extension is already loaded. To reload it, use:

%reload_ext autoreload

## Relevant feature detection

In [2]:
student, env, results = run_experiment(num_games=5, get_context=True, verbose=True)

  0%|          | 0/5 [00:00<?, ?it/s]

1 2 3
X 5 6
7 8 9
Player O selects 9


 20%|██        | 1/5 [00:41<02:46, 41.65s/it]

1 2 3
X 5 6
X 8 O
Player O selects 8
1 2 3
4 5 6
X 8 9
Player O selects 5
1 2 3
X O 6
X 8 9
Player O selects 2


 40%|████      | 2/5 [01:36<02:28, 49.51s/it]

...

X X 3
4 O X
7 O 9
Player O selects 9
1 2 3
4 5 6
7 8 X
Player O selects 5


 80%|████████  | 4/5 [03:08<00:46, 46.29s/it]

1 2 3
4 O X
7 8 X
Player O selects 8
1 2 3
X 5 6
7 8 9
Player O selects 9
1 2 3
X 5 6
X 8 O
Player O selects 8


100%|██████████| 5/5 [04:04<00:00, 48.99s/it]

1 2 3
X X 6
X O O
Player O selects 2


In [3]:
print(student.stats)

{'top_features': {(1, 2, 3, 'X', 5, 6, 7, 8, 9): [FeatureActivations(
   0: (Feature("The number 9 in statistical and numerical sequence contexts"), 1.2109375)
   1: (Feature("The assistant is presenting or describing ordered options or items"), 1.1015625)
   2: (Feature("Technical writing about data structures and programming concepts"), 1.09375)
   3: (Feature("Maximum value in rating scales (typically 10 or 100)"), 1.0859375)
   4: (Feature("The assistant should provide Tic Tac Toe implementation code"), 0.99609375)
), FeatureActivations(
   0: (Feature("The number 9 in statistical and numerical sequence contexts"), 1.2109375)
   1: (Feature("The assistant is presenting or describing ordered options or items"), 1.1015625)
   2: (Feature("Technical writing about data structures and programming concepts"), 1.09375)
   3: (Feature("Maximum value in rating scales (typically 10 or 100)"), 1.0859375)
   4: (Feature("The assistant should provide Tic Tac Toe implementation code"), 0.9960937

In [4]:
# Combine all lists into one
state_features = []
for key in student.stats['top_features']:
    state_features += student.stats['top_features'][key]

all_features = []
for i in range(len(state_features)):
    all_features += state_features[i]
    
feature_values = []
for feature in all_features:
    feature_values.append(feature.feature)

feature_counts = Counter(feature_values)
print(feature_counts)

Counter({Feature("ASCII art game boards and grid representations"): 9, Feature("Comma-separated numbers in structured data contexts"): 7, Feature("The number 8 in statistical and predictive modeling contexts"): 4, Feature("Middle column (B) values in structured data examples"): 4, Feature("Delimiter before final number in lottery-style sequences"): 4, Feature("The number 2 in Python code examples and data structures"): 4, Feature("The number 9 in statistical and numerical sequence contexts"): 3, Feature("The assistant is presenting or describing ordered options or items"): 3, Feature("Technical writing about data structures and programming concepts"): 3, Feature("Maximum value in rating scales (typically 10 or 100)"): 3, Feature("The number 5 in numerical contexts"): 3, Feature("Higher numbers (20-45) in lottery sequences"): 3, Feature("The assistant should provide Tic Tac Toe implementation code"): 2, Feature("The literal number 2"): 2, Feature("Sequences of consecutive small integers

In [6]:
# Save the results
with open('output/results.pkl', 'wb') as f:
    pickle.dump(feature_counts, f)

# Human readable results
df = pd.DataFrame.from_dict(feature_counts, orient='index')
df.to_csv('output/results.csv')

In [13]:
# load the results
with open('output/results.pkl', 'rb') as f:
    test_results = pickle.load(f)
    
print(test_results)

Counter({Feature("ASCII art game boards and grid representations"): 9, Feature("Comma-separated numbers in 
structured data contexts"): 7, Feature("The number 8 in statistical and predictive modeling contexts"): 4, 
Feature("Middle column (B) values in structured data examples"): 4, Feature("Delimiter before final number in 
lottery-style sequences"): 4, Feature("The number 2 in Python code examples and data structures"): 4, Feature("The 
number 9 in statistical and numerical sequence contexts"): 3, Feature("The assistant is presenting or describing 
ordered options or items"): 3, Feature("Technical writing about data structures and programming concepts"): 3, 
Feature("Maximum value in rating scales (typically 10 or 100)"): 3, Feature("The number 5 in numerical contexts"): 
3, Feature("Higher numbers (20-45) in lottery sequences"): 3, Feature("The assistant should provide Tic Tac Toe 
implementation code"): 2, Feature("The literal number 2"): 2, Feature("Sequences of consecutive small integers with
commas"): 2, Feature("HTTP request handling and query parameter access in code"): 2, Feature("Start of a new 
conversation segment, especially for multilingual conversations"): 1, Feature("Epistemological discussions about 
the nature and limitations of knowledge"): 1, Feature("Type declarations and parameter lists in programming code"):
1, Feature("Small integer constants used for array dimensions and buffer sizes in programming"): 1})

## Determining base success rate

In [8]:
student_base, env_base, results_base = run_experiment(num_games=100, get_context=False)
print(results_base)
print(student_base.stats)

100%|██████████| 100/100 [06:43<00:00,  4.03s/it]

None
{'top_features': {}, 'move_5': 83, 'step': 600, 'move_6': 53, 'move_3': 15, 'move_9': 54, 'move_2': 12, 'move_8': 60, 'move_1': 2, 'invalid_move': 10, 'fail_safe': 5, 'move_4': 1, 'move_7': 20}


We define success as winning or drawing. Of course, the tic-tac-toe optimal agent cannot lose, so we only need to look at the draw rate.

## SAE RL

In [14]:
# Sanity checker
from stable_baselines3.common.env_checker import check_env
from tictactoe import TicTacToeSAE
from move_checker import MoveChecker
from agents import OptimalAgent
from constants import TEACHER

move_checker = MoveChecker()
optimal_agent = OptimalAgent(TEACHER, move_checker)

env = TicTacToeSAE(move_checker=move_checker, teacher=optimal_agent)

check_env(env)

Bound: 0.2

/Users/keenanpepper/sae-rl/.venv/lib/python3.12/site-packages/stable_baselines3/common/env_checker.py:462: 
UserWarning: We recommend you to use a symmetric and normalized Box action space (range=[-1, 1]) cf. 
https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html
  warnings.warn(

/Users/keenanpepper/sae-rl/.venv/lib/python3.12/site-packages/stable_baselines3/common/env_checker.py:473: 
UserWarning: Your action space has dtype float64, we recommend using np.float32 to avoid cast errors.
  warnings.warn(

In [16]:
# This might crash
student_rl, env_rl, results_rl = run_experiment(num_games=10000, get_context=False, use_rl_agent=True, test_agent=False) # num_games is really num steps in this case

Bound: 0.2

Using cpu device

Wrapping the env with a `Monitor` wrapper

Wrapping the env in a DummyVecEnv.

Logging to output/tensorboard/SAC_3

LiveError: Only one live display may be active at once

In [ ]:
print(results_rl)

## Testing Agent

In [ ]:
student_test, env_test, results_test = run_experiment(num_games=100, get_context=False, use_rl_agent=True, test_agent=True)

In [ ]:
print(results_test)

In [ ]:
# Look at everything except activations key
for key in env_test.stats.keys():
    if key != 'activations':
        print(key, env_test.stats[key])
        
print("Step", student_test.stats['step'])

In [6]:
pickle.dump(env_test.stats['activations'], open('output/features.pkl', 'wb'))

## Plots

In [ ]:
import matplotlib.pyplot as plt

# Data
best_results = {
    'Baseline': 1,
    'Feature Steering RL': 3,
}

# Create the bar plot
fig, ax = plt.subplots()
ax.bar(best_results.keys(), best_results.values(), color=['blue', 'orange'])

# Set the title and labels
ax.set_title('Draw Rate % Comparison between Agents')
ax.set_xlabel('Method')
ax.set_ylabel('Draw Rate %')

# Enable the grid
#ax.grid(True)

# Show the plot
plt.show()